# First realistic keyframe (GitHub source + Kaggle GPU)

This notebook runs the tracked `pipeline/kaggle_keyframes.py` script for **shot-01 only**, using Gemini image generation with the approved character reference as an image input. The launch cell makes one image-generation request and does not retry automatically. Gemini API quota or billing may apply.

Before running the launch cell, review the shot prompt and make sure you are ready for a model request. The script records the prompt, parameters, image hash, and model call; it does not store credentials.

## 1. Install Kaggle-session dependencies

Enable Kaggle Internet. A GPU is not required for this Gemini API workflow. These packages are installed in the temporary Kaggle session only.

In [ ]:
%pip install -q "google-genai" "jsonschema==4.23.0" "Pillow==11.1.0"

## 2. Configure Gemini API access

This workflow uses the Gemini API, not the Gemini website subscription. Create/use a Gemini API key in Google AI Studio, add it to Kaggle **Add-ons → Secrets** as `GEMINI_API_KEY`, and grant this notebook access. The key stays in Kaggle Secrets; never paste or print it.

In [ ]:
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Add and grant access to the Kaggle secret GEMINI_API_KEY.")
print("Gemini API key loaded from Kaggle Secrets (value hidden).")

## 3. Confirm Gemini API access

The image request sends the approved character reference and scene prompt to Google's Gemini API. Check your API quota and billing before proceeding.

## 4. Generate one Gemini test frame

The launch cell fetches the latest public `main` branch and runs the tracked script for `shot-01` only, passing the API key securely through the subprocess environment. It saves results in a new timestamped folder and leaves previous runs untouched. Running this cell makes one Gemini image-generation request. Review this test frame before generating other shots.

In [ ]:
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/hann2004/ai_cinematic_storytelling.git"
REPOSITORY_DIR = Path("/kaggle/working/ai_cinematic_storytelling")
if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").is_dir():
        raise FileExistsError(f"Path exists but is not a Git clone: {REPOSITORY_DIR}")
    remote_url = subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "remote", "get-url", "origin"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if remote_url != REPOSITORY_URL:
        raise RuntimeError(f"Unexpected Git remote at {REPOSITORY_DIR}: {remote_url}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
artifact_dir = Path("/kaggle/working/keyframe_artifacts") / run_id
artifact_dir.mkdir(parents=True, exist_ok=False)
environment = os.environ.copy()
environment["GEMINI_API_KEY"] = api_key
subprocess.run(
    [
        "python",
        "-m",
        "pipeline.kaggle_keyframes",
        "--shot-id",
        "shot-01",
        "--output-dir",
        str(artifact_dir),
    ],
    cwd=REPOSITORY_DIR,
    env=environment,
    check=True,
)
print("Keyframe code came from GitHub:", REPOSITORY_URL)

## 5. Review and download the result

Review the image before using it. The ZIP contains the image, its metadata, and the one-line model-call record. Do not copy Kaggle secrets into project files.

In [ ]:
from IPython.display import FileLink, display
from PIL import Image
from zipfile import ZIP_DEFLATED, ZipFile

image_path = artifact_dir / "shot-01.png"
display(Image.open(image_path))
download_path = Path.cwd() / f"shot-01-artifacts-{run_id}.zip"
with ZipFile(download_path, "w", ZIP_DEFLATED) as archive:
    for filename in ("shot-01.png", "shot-01.json", "shot-01_run.jsonl"):
        archive.write(artifact_dir / filename, arcname=filename)
display(FileLink(download_path.name))